In [12]:
# Define o endereço do repositório do grupo.
REPO <- "https://raw.githubusercontent.com/andrecesar00/predicarga_andre_diego_kaykhy/main/"

# Carrega as funções da Aula 04.
source("https://raw.githubusercontent.com/fractalarea/pi3-motor-de-busca/main/motor/motor04.R")

# Baixa e carrega a configuração atualizada da Aula 03.
download.file(paste0(REPO, "estrutura/codigo/config.R"), "config.R")
source("config.R")

# Carrega os documentos e seus artigos de origem.
docs <- readRDS(gzcon(url(paste0(REPO, "estrutura/banco-de-dados/docs.rds"))))
origem <- readRDS(gzcon(url(paste0(REPO, "estrutura/banco-de-dados/origem.rds"))))

# Monta o índice usando a limpeza e as stopwords do grupo.
ix <- montar(docs, cfg)

# Mostra a sessão para conferência.
estado()

<!-- estado-R:inicio -->
## Estado do R ao fim da sessão
*gerado por `anexar_estado()` em 2026-10-25 05:35 · R version 4.6.1 (2026-06-24) · motor04 v1 2026-10-01 · não edite à mão*

### Objetos

| objeto | tipo | tamanho | valor / amostra |
|---|---|---|---|
| `avgdl` | numeric | 1 | 62.75 |
| `bm` | numeric, nomeado | 64 | d1 0, d2 0, d3 1.208, d4 3.053, d5 0, d6 0, ... |
| `cfg$minimo` | numeric | 1 | 200 |
| `cfg$limpar` | logical | 1 | TRUE |
| `cfg$acentos` | character | 1 | translit |
| `cfg$stopwords` | character | 12 | de, e, a, o, em, da, ... |
| `cfg$grupo` | character | 1 | Predicarga |
| `consulta` | character | 3 | ponte, pensil, arquitetura |
| `dl` | numeric, nomeado | 64 | d1 58, d2 127, d3 109, d4 152, d5 27, d6 74, ... |
| `docs` | character, nomeado | 64 | d1 São Vicente, oficialmente E..., d2 São Vicente marca o início ..., d3 Com um PIB de mais de 6 bil..., d4 São Vicente preserva divers..., d5 A Ilha de Gohayó foi descob..., d6 Nas primeiras décadas do sé..., ... 

In [13]:
# Conta os tokens de cada documento depois da limpeza.
dl <- colSums(ix$tf)

# Calcula o tamanho médio dos documentos.
avgdl <- mean(dl)

# Calcula o IDF específico do BM25.
idf_b <- log((ix$N - ix$df + 0.5) / (ix$df + 0.5) + 1)

# Mostra o menor tamanho, o maior e a média.
c(
  menor = min(dl),
  maior = max(dl),
  media = round(avgdl, 1)
)

menor maior media 
 25.0 205.0  62.8

In [14]:
# Identifica o documento mais longo.
dl[dl == max(dl)]

# Mostra o artigo de origem dele.
origem[names(dl[dl == max(dl)])]

# Compara seu tamanho com a média e calcula o ajuste.
razao <- max(dl) / avgdl
K <- 1.2 * (1 - 0.75 + 0.75 * razao)

# Calcula o peso relativo de um termo que aparece uma vez.
c(razao = razao, K = K, fracao = 2.2 / (1 + K))

d36 
205

d36 
"Martim Afonso de Sousa"

razao         K    fracao 
3.2669323 3.2402390 0.5188387

In [15]:
# Calcula o escore BM25 de um documento.
bm25_doc <- function(termos, d, k1 = 1.2, b = 0.75) {
  s <- 0                                      # Começa a soma em zero.
  for (t in termos) {
    if (!t %in% ix$vocab) next                 # Pula termos fora do índice.
    f <- ix$tf[t, d]                           # Frequência no documento.
    if (f == 0) next                          # Termo ausente contribui zero.
    K <- k1 * (1 - b + b * dl[[d]] / avgdl)   # Ajusta pelo tamanho.
    s <- s + idf_b[[t]] * (f * (k1 + 1)) / (f + K)
  }
  s                                           # Devolve o escore total.
}

In [16]:
# Define a primeira consulta de trabalho.
q <- "monumentos historicos centro"

# Aplica a mesma preparação usada nos documentos.
consulta <- preparar(q, cfg)

# Calcula o BM25 de cada documento.
bm <- sapply(colnames(ix$tf), function(d) bm25_doc(consulta, d))

# Mostra os cinco primeiros de cada modelo.
list(
  BM25 = round(sort(bm, decreasing = TRUE)[1:5], 3),
  cosseno = round(ranking_cosseno(q, ix, cfg)[1:5], 3)
)

$BM25
   d6    d2    d4    d1    d3 
3.511 2.296 2.060 0.000 0.000 

$cosseno
   d6    d2    d4    d1    d3 
0.113 0.063 0.050 0.000 0.000

In [17]:
# Define e prepara a segunda consulta.
q <- "fundacao vila martim afonso"
consulta <- preparar(q, cfg)

# Calcula o BM25 para cada documento.
bm <- sapply(colnames(ix$tf), function(d) bm25_doc(consulta, d))

# Compara os cinco primeiros dos dois modelos.
list(
  BM25 = round(sort(bm, decreasing = TRUE)[1:5], 3),
  cosseno = round(ranking_cosseno(q, ix, cfg)[1:5], 3)
)

$BM25
  d24   d17    d7   d38    d8 
4.369 4.298 4.023 3.861 3.599 

$cosseno
   d4   d27   d24   d28    d7 
0.098 0.067 0.058 0.055 0.055

In [18]:
# Compara os tamanhos com a média do corpus.
dl[c("d24", "d4")]
avgdl

# Mostra a frequência de cada termo nos dois documentos.
ix$tf[intersect(consulta, ix$vocab), c("d24", "d4")]

d24  d4 
 31 152

[1] 62.75

,d24,d4
fundacao,0,1
vila,1,2
martim,1,0
afonso,1,0


In [19]:
# Define e prepara a terceira consulta.
q <- "ponte pensil arquitetura"
consulta <- preparar(q, cfg)

# Calcula o BM25 de cada documento.
bm <- sapply(colnames(ix$tf), function(d) bm25_doc(consulta, d))

# Mostra os cinco primeiros dos dois modelos.
list(
  BM25 = round(sort(bm, decreasing = TRUE)[1:5], 3),
  cosseno = round(ranking_cosseno(q, ix, cfg)[1:5], 3)
)

$BM25
  d53    d4   d64   d57   d62 
7.058 3.053 2.749 2.718 2.533 

$cosseno
  d53   d64   d57    d4   d62 
0.372 0.113 0.104 0.085 0.076

In [20]:
# Compara os tamanhos dos dois documentos.
dl[c("d4", "d64")]
avgdl

# Mostra quantas vezes cada termo aparece em cada documento.
ix$tf[intersect(consulta, ix$vocab), c("d4", "d64")]

d4 d64 
152  51

[1] 62.75

,d4,d64
ponte,1,4
pensil,1,0


In [21]:
# Consulta 3
q <- "ponte pensil arquitetura"
consulta <- preparar(q, cfg)

bm <- sapply(colnames(ix$tf), function(d) bm25_doc(consulta, d))

list(
  BM25 = round(sort(bm, decreasing = TRUE)[1:5], 3),
  cosseno = round(ranking_cosseno(q, ix, cfg)[1:5], 3)
)

$BM25
  d53    d4   d64   d57   d62 
7.058 3.053 2.749 2.718 2.533 

$cosseno
  d53   d64   d57    d4   d62 
0.372 0.113 0.104 0.085 0.076

In [24]:
# Consulta 1
q <- "monumentos historicos centro"
consulta <- preparar(q, cfg)

bm <- sapply(colnames(ix$tf), function(d) bm25_doc(consulta, d))

list(
  BM25 = round(sort(bm, decreasing = TRUE)[1:5], 3),
  cosseno = round(ranking_cosseno(q, ix, cfg)[1:5], 3)
)

$BM25
   d6    d2    d4    d1    d3 
3.511 2.296 2.060 0.000 0.000 

$cosseno
   d6    d2    d4    d1    d3 
0.113 0.063 0.050 0.000 0.000

In [25]:
# Consulta 2
q <- "fundacao vila martim afonso"
consulta <- preparar(q, cfg)

bm <- sapply(colnames(ix$tf), function(d) bm25_doc(consulta, d))

list(
  BM25 = round(sort(bm, decreasing = TRUE)[1:5], 3),
  cosseno = round(ranking_cosseno(q, ix, cfg)[1:5], 3)
)

$BM25
  d24   d17    d7   d38    d8 
4.369 4.298 4.023 3.861 3.599 

$cosseno
   d4   d27   d24   d28    d7 
0.098 0.067 0.058 0.055 0.055